# Phase 5 — NB2: Stage 2 Sentiment Training (Retrieval + No-Retrieval)

**Goal:** Train Stage 2 (sentiment per category) in 2 variants:
1. With retrieval + label interpolation (832d)
2. Without retrieval baseline (768d)

**Input:**
- `lcminhc/p5-nb1-stage1` — Stage 1 ckpt + processed data (from NB1)
- `lcminhc/p3s2-embedding-flat` — embedding ckpt + FAISS index

**Output:** Upload `/kaggle/working/outputs_p5_nb2/` as Kaggle dataset `p5-nb2-stage2`
- `stage2_best.pt` (retrieval)
- `stage2_noret_best.pt` (no-retrieval)
- Training logs

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml

In [ ]:
import os
import sys
import json
import shutil

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
# Wire NB1 output (Stage 1 ckpt + processed data)
NB1_INPUT = None
for candidate in ['/kaggle/input/p5-nb1-stage1',
                  '/kaggle/input/datasets/lcminhc/p5-nb1-stage1']:
    if os.path.exists(candidate):
        NB1_INPUT = candidate
        break
assert NB1_INPUT, 'Dataset p5-nb1-stage1 not found'
print(f'NB1 input: {NB1_INPUT}')

# Wire embedding + FAISS
EMB_INPUT = None
for candidate in ['/kaggle/input/p3s2-embedding-flat',
                  '/kaggle/input/datasets/lcminhc/p3s2-embedding-flat']:
    if os.path.exists(candidate):
        EMB_INPUT = candidate
        break
assert EMB_INPUT, 'Dataset p3s2-embedding-flat not found'
print(f'Embedding input: {EMB_INPUT}')

# Copy processed data from NB1
os.makedirs('data/processed', exist_ok=True)
for f in ['category_detection.jsonl', 'sentiment_records.jsonl']:
    shutil.copy(f'{NB1_INPUT}/{f}', f'data/processed/{f}')
    print(f'data/processed/{f}')

# Copy embedding checkpoint
os.makedirs('checkpoints/embedding', exist_ok=True)
shutil.copy(f'{EMB_INPUT}/embedding_best.pt', 'checkpoints/embedding/best.pt')
print(f'Embedding ckpt: {os.path.getsize("checkpoints/embedding/best.pt") / 1e6:.1f} MB')

# Copy FAISS index
os.makedirs('indexes', exist_ok=True)
for f in os.listdir(EMB_INPUT):
    if f.endswith(('.faiss', '.npy')) or f == 'train_metadata.jsonl':
        shutil.copy(f'{EMB_INPUT}/{f}', f'indexes/{f}')
        print(f'indexes/{f}')

## 1. Train Stage 2 WITH Retrieval

In [ ]:
import torch
import gc
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!python scripts/04b_train_stage2.py \
    --config configs/stage2.yaml \
    --embedding_ckpt checkpoints/embedding/best.pt \
    --index_dir indexes/ \
    --retrieval_config configs/retrieval_v2.yaml \
    --grad_accum_steps 4

In [ ]:
# Save retrieval checkpoint immediately
output_dir = '/kaggle/working/outputs_p5_nb2'
os.makedirs(output_dir, exist_ok=True)
if os.path.exists('checkpoints/stage2/best.pt'):
    shutil.copy('checkpoints/stage2/best.pt', f'{output_dir}/stage2_best.pt')
    size_mb = os.path.getsize('checkpoints/stage2/best.pt') / 1e6
    print(f'stage2_best.pt (retrieval): {size_mb:.1f} MB')
if os.path.exists('logs/stage2_training.jsonl'):
    os.makedirs(f'{output_dir}/logs', exist_ok=True)
    shutil.copy('logs/stage2_training.jsonl', f'{output_dir}/logs/stage2_training.jsonl')
    print('Retrieval training log saved')

## 2. Train Stage 2 WITHOUT Retrieval (Baseline)

In [ ]:
# Clear GPU before no-retrieval training
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
print('GPU cache cleared.')

In [ ]:
!python scripts/04b_train_stage2.py \
    --config configs/stage2_noret.yaml \
    --no_retrieval \
    --grad_accum_steps 4

## 3. Training Logs

In [ ]:
for label, log_path in [('Retrieval', 'logs/stage2_training.jsonl'),
                        ('No-Retrieval', 'logs/stage2_noret_training.jsonl')]:
    print(f'\n=== {label} ===')
    if not os.path.exists(log_path):
        print('No log found.')
        continue
    print(f'{"Epoch":<8} {"Loss":<10} {"Sent Acc":<10} {"Macro F1":<10}')
    print('-' * 38)
    with open(log_path) as f:
        for line in f:
            rec = json.loads(line)
            print(f"{rec['epoch']:<8} {rec['train_loss']:<10.4f} "
                  f"{rec['sentiment_acc']:<10.4f} {rec['sentiment_macro_f1']:<10.4f}")

## 4. Save Outputs

In [ ]:
output_dir = '/kaggle/working/outputs_p5_nb2'
os.makedirs(output_dir, exist_ok=True)

# Retrieval checkpoint (may already be saved above)
if os.path.exists('checkpoints/stage2/best.pt'):
    shutil.copy('checkpoints/stage2/best.pt', f'{output_dir}/stage2_best.pt')
    print(f'stage2_best.pt: {os.path.getsize("checkpoints/stage2/best.pt") / 1e6:.1f} MB')

# No-retrieval checkpoint
if os.path.exists('checkpoints/stage2_noret/best.pt'):
    shutil.copy('checkpoints/stage2_noret/best.pt', f'{output_dir}/stage2_noret_best.pt')
    print(f'stage2_noret_best.pt: {os.path.getsize("checkpoints/stage2_noret/best.pt") / 1e6:.1f} MB')

# Logs
if os.path.exists('logs'):
    shutil.copytree('logs', f'{output_dir}/logs', dirs_exist_ok=True)
    print('logs/ copied')

print(f'\nOutputs saved to {output_dir}')
print('Upload as Kaggle dataset: p5-nb2-stage2')

In [ ]:
shutil.make_archive('/kaggle/working/outputs_p5_nb2_backup', 'zip',
                    '/kaggle/working', 'outputs_p5_nb2')
size_mb = os.path.getsize('/kaggle/working/outputs_p5_nb2_backup.zip') / 1e6
print(f'Backup zip: {size_mb:.1f} MB')